### Step 1: Create the initial customer dimension

In [0]:
from pyspark.sql import functions as F

customer_data = [
    ("C1001","Raj","Chennai"),
    ("C1002","Priya","Madurai")
]

customer_df = spark.createDataFrame(
    customer_data,
    ['customer_id','customer_name','city']
)

customer_df.show()

+-----------+-------------+-------+
|customer_id|customer_name|   city|
+-----------+-------------+-------+
|      C1001|          Raj|Chennai|
|      C1002|        Priya|Madurai|
+-----------+-------------+-------+



In [0]:
customer_history_df = (
    customer_df
    .withColumn('start_date',F.lit('2026-01-01').cast('date'))
    .withColumn("end_date",F.lit(None).cast("date"))
    .withColumn("is_current",F.lit(True))
)

In [0]:
customer_history_df.write.format("delta").mode("overwrite").saveAsTable(
    'workspace.ecommerce_project.dim_customer_history'
)

### Step 2: Create the new customer information

In [0]:
new_customer_data = [
    ('C1001','Raj','Bengaluru')
]

new_customer_df = spark.createDataFrame(
    new_customer_data,
    ['customer_id','customer_name','city']
)

### Step 3: Identify changed records

In [0]:
current_customer_df = spark.table("workspace.ecommerce_project.dim_customer_history").filter(
    F.col("is_current")==True
)

In [0]:
changed_customer_df = (
    new_customer_df.alias('source')
    .join(
        current_customer_df.alias("target"),
        on="customer_id",
        how="inner"
    )
    .filter(
        ~F.col("source.city")
        .eqNullSafe(F.col("target.city"))
    )
    .select(
        F.col("source.customer_id"),
        F.col("source.customer_name"),
        F.col("source.city")
    )
)

In [0]:
# Materialize changed records into a new DataFrame so they persist
# after the MERGE modifies the source table (Spark DataFrames are lazy)
changed_customer_df = spark.createDataFrame(
    changed_customer_df.collect(),
    changed_customer_df.schema
)
changed_customer_df.show()

+-----------+-------------+---------+
|customer_id|customer_name|     city|
+-----------+-------------+---------+
|      C1001|          Raj|Bengaluru|
+-----------+-------------+---------+



### Step 4: Close the Existing Record

In [0]:
from delta.tables import DeltaTable

customer_target = DeltaTable.forName(
    spark,
    "workspace.ecommerce_project.dim_customer_history"
)

In [0]:
(
    customer_target.alias("target")
    .merge(
        changed_customer_df.alias("source"),
        """
        target.customer_id = source.customer_id
        AND target.is_current = true
        """
    )
    .whenMatchedUpdate(
        set = {
            "end_date":"date_sub(current_date(),1)",
            "is_current":"false"
        }
    )
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### Step 5: Insert the New Version

In [0]:
new_version_df = (
    changed_customer_df
    .withColumn("start_date",F.current_date())
    .withColumn("end_date",F.lit(None).cast("date"))
    .withColumn("is_current",F.lit(True))
)

In [0]:
new_version_df.show()

+-----------+-------------+---------+----------+--------+----------+
|customer_id|customer_name|     city|start_date|end_date|is_current|
+-----------+-------------+---------+----------+--------+----------+
|      C1001|          Raj|Bengaluru|2026-10-09|    NULL|      true|
+-----------+-------------+---------+----------+--------+----------+



In [0]:
new_version_df.write.format("delta").mode("append").saveAsTable(
    "workspace.ecommerce_project.dim_customer_history"
)

In [0]:
%sql
select * from workspace.ecommerce_project.dim_customer_history;

customer_id,customer_name,city,start_date,end_date,is_current
C1002,Priya,Madurai,2026-01-01,null,true
C1001,Raj,Chennai,2026-01-01,2026-10-08,false
C1001,Raj,Bengaluru,2026-10-09,null,true
